# 12 - Evolução do baseline: Isolation Forest com preço relativo ao SKU
**Squad 1 · Data Quality em Tempo Real | ML em Streaming**
**Responsável:** Riuler José Silvério
**Card do Trello:** Evolução do baseline (P1 · Anomalias em pedidos e itens · Modelo A)

> Card: *"Apresentar a evolução do baseline com resultados que comprovam a melhora. Para modelos não supervisionados, trazer exemplos que estejam fazendo mais sentido após mudanças, como um registro claramente anômalo."*

### Ponto de partida (notebook `10`)
A baseline usa 7 features de item e pedido e tem, no teste, Precision 0,75 · Recall 0,59 · F1 0,66 · PR-AUC 0,74, com 98 alertas sem regra. A análise dela deixou três pistas:
1. **Os melhores achados eram preços absurdos para o produto** (ração 10 kg a R$ 5.292), mas o modelo só via o preço **absoluto**: uma ração de R$ 300 e um sabão de R$ 300 pareciam iguais.
2. **As features de pedido geravam alertas por estrutura do pedido** (pedido de 1 item, kits de 12 refrigerantes), e não por erro do item.
3. **A versão foi escolhida olhando o teste**, então as métricas da baseline são um pouco otimistas.

### O que muda
| Mudança | Por quê |
|---|---|
| **Nova feature `rel_preco_sku`** = preço ÷ preço típico do SKU (em log) | Compara o preço com o do próprio produto. O típico é a mediana do SKU no treino, usando só itens que passaram nas regras e SKUs com 3 ou mais itens; senão, a mediana da categoria (prefixo do SKU) |
| **Sai o preço absoluto e saem as features de pedido** | Testado na validação: o preço relativo substitui o absoluto, e as features de pedido atrapalham |
| **`max_features = 0,5`** | Cada árvore usa metade das features, e nenhuma feature sozinha domina o score |
| **Escolha feita numa validação** (treino até 05/10 e validação em 07/10) | O teste (08/10 e 15/10) é usado uma vez só, no fim, para as duas versões |

O rótulo fraco continua **fora do treino**. Ele é usado só para avaliar e para limpar a referência de preço com as regras da Silver, informação que já existe em produção para cada linha.

In [0]:
# Credenciais via .env (nunca no código). O .env fica fora do Git (.gitignore).
import os
from dotenv import load_dotenv

CAMINHOS_ENV = [".env", "../.env", "../../estagio-empregadados-turma-2/.env"]
caminho_env = next((p for p in CAMINHOS_ENV if os.path.exists(p)), None)
if caminho_env:
    load_dotenv(caminho_env)

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
assert all([client_id, tenant_id, client_secret, storage_account]), "Credenciais do ADLS não encontradas no .env"

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

base_squad1 = f"abfss://squad1@{storage_account}.dfs.core.windows.net"
caminhos = {
    "features_itens": f"{base_squad1}/dupla1/ml/features_itens_pedido",        # entrada (notebook 09)
    "scores_itens_v2": f"{base_squad1}/dupla1/ml/scores_itens_pedido_v2",      # saída deste notebook
}
print("Credenciais carregadas do .env e caminhos prontos.")

### Etapa 1 — Leitura e divisões
- **Treino:** lotes até 07/10 (8.075 itens), o mesmo da baseline.
- **Validação:** dentro do treino, o último dia (07/10) fica separado para escolher as variantes. Nessa etapa, o ajuste usa só os lotes até 05/10.
- **Teste:** lotes de 08/10 e 15/10, o mesmo da baseline, usado uma única vez.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = (spark.read.format("delta").options(**adls_options).load(caminhos["features_itens"])
      .toPandas().sort_values(["data_lote", "lote", "id_item_pedido"]).reset_index(drop=True))

In [0]:
AZUL, VERMELHO, AMARELO, CINZA, VERDE = "#1F4E8C", "#C0392B", "#F2B705", "#7F8C8D", "#2E7D32"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})

df["data_lote"] = pd.to_datetime(df["data_lote"])
df["categoria"] = df["sku"].str.split("-").str[0]
CORTE_VALIDACAO = df.loc[df.split == "treino", "data_lote"].max()   # último dia do treino (07/10)

treino, teste = df[df.split == "treino"], df[df.split == "teste"]
ajuste_val, validacao = treino[treino.data_lote < CORTE_VALIDACAO], treino[treino.data_lote == CORTE_VALIDACAO]
for nome, base in [("treino", treino), ("  ajuste (validação)", ajuste_val), ("  validação", validacao), ("teste", teste)]:
    print(f"{nome:<22} {len(base):>6,} itens | rótulo fraco {base.rotulo_fraco.mean():.2%}")

### Etapa 2 — Nova feature: preço relativo ao típico do SKU
`rel_preco_sku = log(1 + preço) − log(1 + preço típico do SKU)`. Ela vale perto de 0 para um preço normal, fica bem positiva para um preço muito acima do típico e bem negativa para preço zero ou negativo.

**Como o preço típico é calculado, só com a base de ajuste (sem olhar validação ou teste):**
1. Usa só os itens que **passaram em todas as regras** da etapa 1, para que preços zero ou negativos não entrem na referência.
2. Mediana do SKU, se o SKU tem **3 ou mais itens**. Com 2 itens, um único preço absurdo (que nenhuma regra pega) puxa a mediana; com 3, ele é ignorado.
3. Senão, usa a mediana da **categoria** (prefixo do SKU); senão, a mediana geral.

A célula abaixo também compara as três formas de calcular a referência que foram testadas. A coluna da direita conta os itens normais cujo "típico" saiu mais de 10 vezes acima do próprio preço, sinal de que a referência foi contaminada por um preço absurdo.

> **Escolha: 3 ou mais itens.** O mínimo de 2 itens tem PR-AUC um pouco maior na validação, mas deixa o dobro de referências contaminadas. Como a validação tem só 1.644 itens, a escolha segue o critério de **robustez**, e não a maior métrica.

In [0]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, precision_recall_curve

def referencia_preco(base_ref, x, min_itens=3, so_limpos=True):
    """Acrescenta rel_preco_sku a x, com o preço típico calculado em base_ref."""
    x = x.copy()
    ref_base = base_ref[base_ref.rotulo_fraco == 0] if so_limpos else base_ref
    g_sku, g_cat = ref_base.groupby("sku").preco_unitario, ref_base.groupby("categoria").preco_unitario
    med_sku = g_sku.median()[g_sku.size() >= min_itens]
    med_cat = g_cat.median()[g_cat.size() >= min_itens]
    preco_tipico = x.sku.map(med_sku).fillna(x.categoria.map(med_cat)).fillna(ref_base.preco_unitario.median())
    x["preco_tipico"] = preco_tipico.round(2)
    x["origem_referencia"] = np.where(x.sku.isin(med_sku.index), "sku", np.where(x.categoria.isin(med_cat.index), "categoria", "geral"))
    x["rel_preco_sku"] = x.log_preco - np.log1p(preco_tipico.clip(lower=0))
    return x

ITEM = ["log_preco", "quantidade", "razao_desconto", "log_valor_liquido_item"]
PEDIDO = ["n_itens_pedido", "n_skus_pedido", "log_valor_liquido_pedido"]
VERSOES = {
    "baseline (7)": ITEM + PEDIDO,
    "7 + rel_preco": ITEM + PEDIDO + ["rel_preco_sku"],
    "item + rel_preco": ITEM + ["rel_preco_sku"],
    "item c/ rel_preco no lugar do preço": ["rel_preco_sku"] + ITEM[1:],
}
SEMENTES = range(5)

def treinar(base, features, max_features=1.0, contamination=0.09, semente=42):
    return IsolationForest(n_estimators=300, max_samples=256, max_features=max_features,
                           contamination=contamination, random_state=semente, n_jobs=-1).fit(base[features])

def pr_auc_sementes(base_fit, base_aval, features, max_features):
    v = [average_precision_score(base_aval.rotulo_fraco, -treinar(base_fit, features, max_features, semente=s).score_samples(base_aval[features]))
         for s in SEMENTES]
    return np.mean(v), np.std(v)

# Três formas de calcular a referência (versão "item c/ rel_preco no lugar do preço", max_features 0,5)
feats = VERSOES["item c/ rel_preco no lugar do preço"]
linhas = []
for nome, kw in [("mediana do SKU, todos os itens", dict(min_itens=1, so_limpos=False)),
                 ("SKU só com itens limpos, 2+ itens", dict(min_itens=2)),
                 ("SKU só com itens limpos, 3+ itens", dict(min_itens=3))]:
    a, v = referencia_preco(ajuste_val, ajuste_val, **kw), referencia_preco(ajuste_val, validacao, **kw)
    m, d = pr_auc_sementes(a, v, feats, 0.5)
    linhas.append({"referência": nome, "PR-AUC validação (média ± dp)": f"{m:.3f} ± {d:.3f}",
                   "itens limpos com típico > 10× o preço": int(((v.rel_preco_sku < np.log(0.1)) & (v.rotulo_fraco == 0)).sum())})
display(pd.DataFrame(linhas))

### Etapa 3 — Escolha na validação
Cada combinação de features e `max_features` é treinada com 5 sementes na base de ajuste e medida pela **PR-AUC na validação**. A PR-AUC não depende do limiar. A melhor combinação segue para o teste.

In [0]:
ajuste_r = referencia_preco(ajuste_val, ajuste_val)
validacao_r = referencia_preco(ajuste_val, validacao)

linhas = []
for nome, feats in VERSOES.items():
    for mf in [1.0, 0.5]:
        m, d = pr_auc_sementes(ajuste_r, validacao_r, feats, mf)
        linhas.append({"versão": nome, "max_features": mf, "nº features": len(feats), "PR-AUC média": m, "desvio": d})
selecao = pd.DataFrame(linhas).sort_values("PR-AUC média", ascending=False).reset_index(drop=True)
display(selecao.round(3))

VERSAO_NOVA, MF_NOVA = selecao.loc[0, "versão"], selecao.loc[0, "max_features"]
FEATS_NOVA = VERSOES[VERSAO_NOVA]
print(f"Escolhida: {VERSAO_NOVA} | max_features = {MF_NOVA}")

#### Limiar (`contamination`)
O `contamination` define quantos itens viram alerta, ou seja, o **volume de revisão**. Ele é escolhido pelo **maior F1 na validação**.

In [0]:
linhas = []
for c in [0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10]:
    m = treinar(ajuste_r, FEATS_NOVA, MF_NOVA, c)
    alerta = (m.predict(validacao_r[FEATS_NOVA]) == -1).astype(int)
    y = validacao_r.rotulo_fraco
    linhas.append({"contamination": c, "% alertados": alerta.mean() * 100, "Precision": precision_score(y, alerta, zero_division=0),
                   "Recall": recall_score(y, alerta), "F1": f1_score(y, alerta)})
limiares = pd.DataFrame(linhas)
display(limiares.round(3))
CONTAMINATION_NOVA = float(limiares.loc[limiares.F1.idxmax(), "contamination"])
print(f"contamination escolhido: {CONTAMINATION_NOVA}")

### Etapa 4 — Teste: baseline × evolução
As duas versões são treinadas com **todo o treino** (até 07/10) e avaliadas **uma vez** no teste. A baseline é exatamente a do notebook `10` (7 features, `contamination = 0,09`, `random_state = 42`). A referência de preço da evolução é recalculada com todo o treino.

In [0]:
treino_r, teste_r = referencia_preco(treino, treino), referencia_preco(treino, teste)
MODELOS = {
    "baseline": dict(feats=VERSOES["baseline (7)"], mf=1.0, c=0.09),
    "evolução": dict(feats=FEATS_NOVA, mf=MF_NOVA, c=CONTAMINATION_NOVA),
}

def metricas(score, alerta, y):
    return {"% alertados": alerta.mean() * 100, "Precision": precision_score(y, alerta, zero_division=0),
            "Recall": recall_score(y, alerta), "F1": f1_score(y, alerta), "PR-AUC": average_precision_score(y, score),
            "alertas sem regra": int(((alerta == 1) & (y == 0)).sum())}

modelos, linhas = {}, []
for nome, cfg in MODELOS.items():
    modelos[nome] = treinar(treino_r, cfg["feats"], cfg["mf"], cfg["c"])
    teste_r[f"score_{nome}"] = -modelos[nome].score_samples(teste_r[cfg["feats"]])
    teste_r[f"alerta_{nome}"] = (modelos[nome].predict(teste_r[cfg["feats"]]) == -1).astype(int)
    linhas.append({"versão": nome, **metricas(teste_r[f"score_{nome}"], teste_r[f"alerta_{nome}"], teste_r.rotulo_fraco)})
resultado = pd.DataFrame(linhas)
display(resultado.round(3))

# Mesma semente para as duas versões: a diferença de PR-AUC não depende da sorte do sorteio das árvores
pares = []
for s in SEMENTES:
    pr = {nome: average_precision_score(teste_r.rotulo_fraco, -treinar(treino_r, cfg["feats"], cfg["mf"], cfg["c"], s).score_samples(teste_r[cfg["feats"]]))
          for nome, cfg in MODELOS.items()}
    pares.append({"semente": s, **pr, "ganho": pr["evolução"] - pr["baseline"]})
pares = pd.DataFrame(pares)
display(pares.round(3))
print(f"Ganho de PR-AUC: {pares.ganho.mean():+.3f} em média | positivo em {(pares.ganho > 0).sum()} de {len(pares)} sementes | "
      f"aleatório: {teste_r.rotulo_fraco.mean():.3f}")

#### Mesmo volume de alertas
Para comparar no **mesmo esforço de revisão**, a evolução também é medida com o limiar que alerta a mesma fração do teste que a baseline.

In [0]:
pct_base = teste_r.alerta_baseline.mean()
corte = np.quantile(teste_r["score_evolução"], 1 - pct_base)
alerta_mesmo_volume = (teste_r["score_evolução"] > corte).astype(int)
mesmo_volume = pd.DataFrame([
    {"versão": "baseline", **metricas(teste_r.score_baseline, teste_r.alerta_baseline, teste_r.rotulo_fraco)},
    {"versão": "evolução (mesmo volume)", **metricas(teste_r["score_evolução"], alerta_mesmo_volume, teste_r.rotulo_fraco)},
])
display(mesmo_volume.round(3))

### Etapa 5 — Curva Precision × Recall no teste

In [0]:
fig, ax = plt.subplots(figsize=(8, 5))
for nome, cor in [("baseline", CINZA), ("evolução", AZUL)]:
    p, r, _ = precision_recall_curve(teste_r.rotulo_fraco, teste_r[f"score_{nome}"])
    ax.plot(r, p, color=cor, lw=2, label=f"{nome} (PR-AUC {average_precision_score(teste_r.rotulo_fraco, teste_r[f'score_{nome}']):.2f})")
ax.axhline(teste_r.rotulo_fraco.mean(), color=VERMELHO, ls="--", lw=1, label="aleatório")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision × Recall no teste")
ax.legend()
plt.show()

### Etapa 6 — Recall por regra

In [0]:
REGRAS = {"quantidade 0": teste_r.quantidade < 1, "preço ≤ 0": teste_r.preco_unitario <= 0,
          "desconto > preço": teste_r.desconto_aplicado > teste_r.preco_unitario, "desconto negativo": teste_r.desconto_aplicado < 0}
teste_r["regra"] = np.select(list(REGRAS.values()), list(REGRAS.keys()), "nenhuma")
display(pd.DataFrame([{"regra": k, "itens com falha": int(m.sum()),
                       **{f"Recall {nome}": teste_r.loc[m, f"alerta_{nome}"].mean() if m.any() else np.nan for nome in MODELOS}}
                      for k, m in REGRAS.items()]).round(3))

### Etapa 7 — Exemplos: o que passou a fazer mais sentido
O card pede exemplos concretos. A tabela cruza os alertas das duas versões. Em seguida vêm quatro grupos: `rank` 1 é o item mais suspeito do teste, e `× típico` é o preço dividido pelo preço típico do SKU.

In [0]:
teste_r["rank_baseline"] = teste_r.score_baseline.rank(ascending=False).astype(int)
teste_r["rank_evolução"] = teste_r["score_evolução"].rank(ascending=False).astype(int)
teste_r["× típico"] = ((1 + teste_r.preco_unitario) / (1 + teste_r.preco_tipico)).round(1)
COLS = ["sku", "quantidade", "preco_unitario", "desconto_aplicado", "preco_tipico", "× típico", "n_itens_pedido", "regra", "rank_baseline", "rank_evolução"]

display(pd.crosstab([teste_r.rotulo_fraco.rename("rótulo fraco"), teste_r.alerta_baseline.rename("alerta baseline")],
                    teste_r["alerta_evolução"].rename("alerta evolução")))

def mostrar(titulo, mask, ordem, n=8):
    x = teste_r[mask].sort_values(ordem)
    print(f"\n{titulo}: {len(x)} itens")
    display(x[COLS].head(n))

mostrar("A) Preços muito acima do típico (> 10×), sem regra quebrada", (teste_r["× típico"] > 10) & (teste_r.rotulo_fraco == 0), "rank_evolução")
mostrar("B) Erros de regra que só a evolução pega", (teste_r.rotulo_fraco == 1) & (teste_r["alerta_evolução"] == 1) & (teste_r.alerta_baseline == 0), "rank_evolução")
mostrar("C) Alertas da baseline sem regra que a evolução deixou de dar", (teste_r.rotulo_fraco == 0) & (teste_r.alerta_baseline == 1) & (teste_r["alerta_evolução"] == 0), "rank_baseline")
mostrar("D) Erros de regra que só a baseline pega", (teste_r.rotulo_fraco == 1) & (teste_r.alerta_baseline == 1) & (teste_r["alerta_evolução"] == 0), "rank_baseline")

### Etapa 8 — Estabilidade no streaming
O modelo pontua cada micro-lote do teste separadamente. O % alertado precisa ficar estável nos lotes com 50 itens ou mais.

In [0]:
por_lote = (teste_r.groupby("lote").agg(itens=("id_item_pedido", "size"), baseline=("alerta_baseline", "mean"), evolucao=("alerta_evolução", "mean"),
                                        rotulo=("rotulo_fraco", "mean")).query("itens >= 50") * [1, 100, 100, 100])
display(por_lote.round(2).sort_values("evolucao"))
print(f"Lotes com 50+ itens: {len(por_lote)} | % alertados na evolução: {por_lote.evolucao.min():.1f}% a {por_lote.evolucao.max():.1f}% "
      f"| baseline: {por_lote.baseline.min():.1f}% a {por_lote.baseline.max():.1f}%")

### Etapa 9 — Gravação dos scores da evolução
Grava score, alerta e a referência de preço de **todos os itens** (treino e teste), pontuados pela evolução, em `squad1/dupla1/ml/scores_itens_pedido_v2`. O caminho é **separado** da baseline, que continua em `scores_itens_pedido`. A tabela é derivada e recalculada a cada execução (`overwrite`).

In [0]:
import json
from datetime import datetime

todos = referencia_preco(treino, df)
modelo = modelos["evolução"]
scores = todos[["id_item_pedido", "id_pedido", "sku", "lote", "data_lote", "split", "rotulo_fraco",
                "quantidade", "preco_unitario", "desconto_aplicado", "preco_tipico", "origem_referencia", "rel_preco_sku"]].copy()
scores["score_anomalia"] = -modelo.score_samples(todos[FEATS_NOVA])
scores["alerta"] = (modelo.predict(todos[FEATS_NOVA]) == -1).astype(int)
scores["limiar"] = -modelo.offset_
scores["versao_modelo"] = f"isolation_forest v2 | {VERSAO_NOVA} | max_features={MF_NOVA} | contamination={CONTAMINATION_NOVA}"
scores["pontuado_em"] = datetime.now()

(spark.createDataFrame(scores).write.format("delta").options(**adls_options)
    .mode("overwrite").option("overwriteSchema", "true").save(caminhos["scores_itens_v2"]))
print(f"[OK] {len(scores):,} scores gravados em {caminhos['scores_itens_v2']}")

m = resultado.set_index("versão").loc["evolução"]
dbutils.notebook.exit(json.dumps({"versao": VERSAO_NOVA, "max_features": MF_NOVA, "contamination": CONTAMINATION_NOVA,
                                  **{k: round(float(m[k]), 4) for k in ["% alertados", "Precision", "Recall", "F1", "PR-AUC"]}}))

### Etapa 10 — Conclusões

> Execução de 22/11, com o mesmo treino (até 07/10, 8.075 itens) e o mesmo teste (08/10 e 15/10, 6.121 itens, 8,35% com rótulo fraco) da baseline. A PR-AUC de um modelo aleatório é 0,083.

**O que mudou:** a feature `rel_preco_sku` (preço ÷ preço típico do SKU) entrou no lugar do preço absoluto, as features de pedido saíram e o modelo passou a usar `max_features = 0,5`. Ficaram **4 features**: `rel_preco_sku`, `quantidade`, `razao_desconto` e `log_valor_liquido_item`. O `contamination = 0,08` foi escolhido pelo maior F1 na validação.

**Resultado no teste**
| Versão | % alertados | Precision | Recall | F1 | PR-AUC | Alertas sem regra |
|---|---|---|---|---|---|---|
| Baseline (notebook 10) | 6,5% | 0,754 | 0,587 | 0,660 | 0,741 | 98 |
| **Evolução** (contamination 0,08) | 7,9% | 0,742 | **0,705** | **0,723** | **0,808** | 125 |
| **Evolução com o mesmo volume da baseline** | 6,5% | **0,809** | **0,630** | **0,708** | **0,808** | **76** |

**Recall por regra no teste**
| Regra | Baseline | Evolução |
|---|---|---|
| Desconto > preço | 100% | 100% |
| Desconto negativo | 75% | **100%** |
| Preço ≤ 0 | 56% | **97%** |
| Quantidade 0 | 14% | **26%** |

**Conclusões**
1. **A melhora se confirma.** A PR-AUC sobe de 0,741 para 0,808, e o ganho aparece em **5 de 5 sementes**, com média de +0,056 quando as duas versões usam a mesma semente. Com o **mesmo volume de alertas** da baseline, a evolução ganha em todas as métricas: Precision de 0,754 para 0,809, Recall de 0,587 para 0,630 e alertas sem regra de 98 para 76.
2. **O ganho vem do preço relativo.** Na validação, a troca do preço absoluto por `rel_preco_sku` leva a PR-AUC de 0,688 para 0,855. Agora o modelo compara o preço com o do próprio produto: R$ 3.367 numa ração cujo típico é R$ 129 (26×) é anômalo, enquanto R$ 300 numa ração de 15 kg não é.
3. **Exemplos que passaram a fazer sentido:**
   - **Preço ≤ 0 passou de 56% para 97% de recall.** Um guardanapo a R$ 0,00, cujo típico é R$ 48,80, subiu da posição 545 para a 253 no ranking de suspeitos.
   - **Os preços absurdos continuam no topo, agora explicados pelo "× típico":** ração Golden a 26× o típico (posição 18), molho Pomarola a 65× (posição 54), Bombril a 103× (posição 100).
   - **Saíram alertas que eram só estrutura do pedido** (33 itens), como o tempero Knorr de R$ 3,18 num pedido de 1 item (posição 172 → 1.772) e o kit de 6 refrigerantes Coca-Cola de 1,5 l a preço normal (254 → 1.043).
4. **O custo é que a evolução não usa mais o contexto do pedido.** Ela deixa de pegar 13 itens com quantidade 0 em **pedidos de 1 ou 2 itens**, que a baseline achava por serem pedidos raros. Mesmo assim, o saldo é positivo: são 73 erros de regra que só a evolução pega, contra 13 que só a baseline pega.
5. **Estabilidade no streaming:** nos lotes com 50 itens ou mais, o % alertado varia de 0% a 13,3% (baseline: 0% a 12,4%) e acompanha o % de rótulo fraco de cada lote. Não há sinal de mudança nos dados que peça retreino.

**Limitações**
- A validação é um único dia (1.644 itens). As escolhas foram conferidas com 5 sementes, mas o ideal é revalidar quando chegarem lotes novos.
- A referência de preço depende do histórico do SKU. Um SKU novo usa a mediana da categoria, que é mais grosseira porque mistura tamanhos de embalagem (no teste, cerca de 40% dos itens usam a categoria). Ela deve ser recalculada a cada retreino.
- A referência usa as regras da Silver só para **limpar** o histórico de preços. O rótulo fraco continua fora do treino.
- O rótulo fraco não é a verdade absoluta: os alertas sem regra com preço dezenas de vezes acima do típico são, muito provavelmente, erros reais.

**Para a comparação com o Autoencoder:** a evolução grava em `scores_itens_pedido_v2`, e a baseline continua em `scores_itens_pedido`. Use o mesmo teste, as mesmas métricas e o mesmo rótulo fraco.
